In [50]:
import os, secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

### QKD Simulation (BB84) (with Eve)

In [51]:
def bb84_simulate(n_bits=127):
    #Alice prepares random bits and bases
    abits = [secrets.randbits(1) for _ in range(n_bits)]        #cryptographically secure random bits.
    abases = [secrets.randbits(1) for _ in range(n_bits)]

    #Eve intercepts and measures in random bases
    ebases = [secrets.randbits(1) for _ in range(n_bits)]
    egoodbits = [
        abits[i] if ebases[i] == abases[i] else secrets.randbelow(2)
        for i in range(n_bits)
    ]

    #Eve resends qubits to Bob
    bbases = [secrets.randbits(1) for _ in range(n_bits)]
    bgoodbits = [
        egoodbits[i] if bbases[i] == ebases[i] else secrets.randbelow(2)
        for i in range(n_bits)
    ]

    #Alice & Bob compare their bases (sifting)
    sifted_indices = [i for i in range(n_bits) if abases[i] == bbases[i]]
    asifted = [abits[i] for i in sifted_indices]
    bsifted = [bgoodbits[i] for i in sifted_indices]

    # Step 5: Calculate error rate introduced by Eve
    errors = sum(1 for a, b in zip(asifted, bsifted) if a != b)
    error_rate = errors / len(asifted) if asifted else 1.0
    return asifted, bsifted, error_rate

In [52]:
def bits_to_bytes(bits):
    b = bytearray()
    for i in range(0, len(bits), 8):
        byte = 0
        for j in range(8):
            if i + j < len(bits):
                byte = (byte << 1) | bits[i + j]
            else:
                byte = (byte << 1)
        b.append(byte)
    return bytes(b)


In [53]:
def derive_key(bits):
    ikm = bits_to_bytes(bits)
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=None,
        info=b"qkd-key",
        backend=default_backend()
    )
    return hkdf.derive(ikm)

### AES-GCM (Used to encryot the PT)

In [54]:
plaintext = b"This is a secret message that I live in IIT Ropar"
print("QKD with Eavesdropper")

abits, bbits, error_rate = bb84_simulate(256)
print(f"Error rate detected between Alice and Bob: {error_rate:.2%}")

QKD WITH EAVESDROPPER (EVE)
Error rate detected between Alice and Bob: 29.46%


In [55]:
#Derive separate keys for Alice and Bob
alice_key = derive_key(abits)
bob_key = derive_key(bbits)

In [56]:
#Compare keys
if alice_key == bob_key:
    print("Keys match — channel secure")
else:
    print("Keys differ — eavesdropper detected.")

Keys differ — eavesdropper detected! Decryption will fail.


In [57]:
#Alice encrypts with her key
aesgcm_alice = AESGCM(alice_key)
nonce = os.urandom(12)
ciphertext = aesgcm_alice.encrypt(nonce, plaintext, None)
print("Ciphertext (truncated):", ciphertext.hex()[:40], "...")

Ciphertext (truncated): e3a371afb9706f780c420abd0ffa7aff6ebce470 ...


In [58]:
#Bob attempts to decrypt with his key
try:
    aesgcm_bob = AESGCM(bob_key)
    decrypted = aesgcm_bob.decrypt(nonce, ciphertext, None)
    print("Decrypted:", decrypted.decode())
except Exception as e:
    print("Decryption failed due to mismatched keys.")

Decryption failed due to mismatched keys (Eve interference).


In [ ]:
#Here, Eve always intercepts the qubits and measures them in random bases.
#Because of the quantum no-cloning principle, her measurement disturbs the qubits and causes errors.
#When Alice and Bob compare a sample of bits, they find an error rate around 25%, which immediately reveals Eve’s presence.
# This is how QKD detects eavesdropping — something impossible in classical key exchange.”